# Part 2: MobileNetV3-Small Model Improvement Ablations

This notebook keeps the Part 2 workflow notebook-owned while moving reusable training and result logic into project modules. The regular-training baseline is loaded from Part 1 results; the remaining ablations are trained here.

In [ ]:
from pathlib import Path
import importlib
import os
import sys


In [ ]:
from pathlib import Path
import importlib
import importlib.util

_REQUIRED_PROJECT_FILES = (
    Path('src/__init__.py'),
    Path('src/utils/notebook_setup.py'),
    Path('src/evaluation/experiment_results.py'),
    Path('src/experiments/enhanced_confidence.py'),
)

_CANDIDATE_PROJECT_ROOTS = (
    Path('/content/drive/MyDrive/MLDS_Final_Project'),
    Path('/content/MLDS_Final_Project'),
    Path('/content/drive/MyDrive/Colab Notebooks/MLDS_Final_Project'),
)


def _safe_resolve(path):
    try:
        return Path(path).resolve()
    except OSError:
        return None


def _safe_exists(path):
    try:
        return Path(path).exists()
    except OSError:
        return False


def _safe_cwd():
    try:
        return Path.cwd().resolve()
    except OSError:
        fallback = Path('/content')
        return fallback if _safe_exists(fallback) else Path.home()


def _safe_glob(path, pattern):
    try:
        return list(path.glob(pattern))
    except OSError:
        return []


def _path_looks_like_project_root(path):
    try:
        return all((path / required).is_file() for required in _REQUIRED_PROJECT_FILES)
    except OSError:
        return False


def _candidate_project_roots():
    current = _safe_cwd()
    candidates = [current, *current.parents, *_CANDIDATE_PROJECT_ROOTS]
    my_drive = Path('/content/drive/MyDrive')
    if _safe_exists(my_drive):
        candidates.extend(_safe_glob(my_drive, 'MLDS_Final_Project'))
        candidates.extend(_safe_glob(my_drive, '*/MLDS_Final_Project'))
    return candidates


def _find_project_root():
    seen = set()
    for candidate in _candidate_project_roots():
        candidate = _safe_resolve(candidate)
        if candidate is None or candidate in seen:
            continue
        seen.add(candidate)
        if _path_looks_like_project_root(candidate):
            return candidate
    return None


def _mount_colab_drive_if_available():
    try:
        from google.colab import drive  # type: ignore
    except ImportError:
        return
    drive.mount('/content/drive', force_remount=True)


_PROJECT_ROOT = _find_project_root()
if _PROJECT_ROOT is None:
    _mount_colab_drive_if_available()
    _PROJECT_ROOT = _find_project_root()

if _PROJECT_ROOT is None:
    raise ModuleNotFoundError(
        'Could not find the full MLDS_Final_Project repo. Expected '
        'src/__init__.py, src/utils/notebook_setup.py, and '
        'src/evaluation/experiment_results.py, and '
        'src/experiments/enhanced_confidence.py. In Colab, upload or clone '
        'the full repo, or place it at /content/drive/MyDrive/MLDS_Final_Project.'
    )

_COLAB_UTILS_PATH = _PROJECT_ROOT / 'src' / 'utils' / 'colab.py'
_COLAB_SPEC = importlib.util.spec_from_file_location('_mlds_colab_bootstrap', _COLAB_UTILS_PATH)
if _COLAB_SPEC is None or _COLAB_SPEC.loader is None:
    raise ImportError(f'Could not load Colab bootstrap helpers from {_COLAB_UTILS_PATH}')
_colab_bootstrap = importlib.util.module_from_spec(_COLAB_SPEC)
_COLAB_SPEC.loader.exec_module(_colab_bootstrap)

ROOT = _colab_bootstrap.bootstrap_notebook_runtime(_PROJECT_ROOT, force_remount=False)
notebook_setup = importlib.import_module('src.utils.notebook_setup')
ROOT


In [ ]:
import pandas as pd
from IPython.core.display import Image
from IPython.display import display

import src.evaluation.experiment_results as experiment_results

experiment_results = importlib.reload(experiment_results)


In [ ]:
import src.experiments.part2 as part2_experiments
import src.experiments.enhanced_confidence as enhanced_confidence

part2_experiments = importlib.reload(part2_experiments)
enhanced_confidence = importlib.reload(enhanced_confidence)
load_part1_model_baseline_aggregated = experiment_results.load_part1_model_baseline_aggregated
stage_configured_colab_data_dir = experiment_results.stage_configured_colab_data_dir
run_part2_improvement_experiments = part2_experiments.run_part2_improvement_experiments
run_part2_enhanced_confidence_experiment = enhanced_confidence.run_part2_enhanced_confidence_experiment
enhanced_confidence_output_paths = enhanced_confidence.enhanced_confidence_output_paths


## Configuration

In [ ]:
part2_setup = notebook_setup.setup_part2_config()
config = part2_setup.config
config.stage_colab_data_to_local_disk = True  # Set False on Colab to read directly from Drive instead of copying to /content.
part2_setup.summary['stage_colab_data_to_local_disk'] = config.stage_colab_data_to_local_disk
part2_setup.summary['colab_local_data_dir'] = config.colab_local_data_dir
device = part2_setup.device
output_paths = part2_setup.output_paths
part2_setup.summary


In [ ]:
print(config)


In [ ]:
%%time
config.data_dir = stage_configured_colab_data_dir(config)
print(f'Active data_dir: {config.data_dir}')


In [ ]:
pd.DataFrame(config.ablations)

## Part 1 MobileNetV3-Small Baseline

In [ ]:
part1_baseline_aggregated = load_part1_model_baseline_aggregated(config, config.model_name)
if not part1_baseline_aggregated.empty:
    display(part1_baseline_aggregated)

## Train Improvement Ablations

In [ ]:
RUN_TRAINING = True

if RUN_TRAINING:
    part2_results = run_part2_improvement_experiments(
        config=config,
        device=device,
        intermediate_figure_callback=lambda path: display(Image(filename=str(path))),
    )
    display(part2_results)
else:
    print('Training is skipped. Set RUN_TRAINING = True to train Part 2 ablations in this notebook.')


## Enhanced Confidence Experiment
Run the reviewer-requested MobileNetV3-Small curriculum permutation-difficulty sweep. It uses the enhanced Part 1 MobileNet baseline as `regular_part1` and stores separate enhanced Part 2 outputs.


In [ ]:
%%time
RUN_ENHANCED_PART2 = True
part2_enhanced_output_paths = enhanced_confidence_output_paths(
    config.results_dir,
    config.figures_dir,
    'part2',
)

if RUN_ENHANCED_PART2:
    part2_enhanced_results = run_part2_enhanced_confidence_experiment(config, device=device)
    display(part2_enhanced_results)
else:
    enhanced_aggregated_path = Path(part2_enhanced_output_paths['aggregated_results'])
    if enhanced_aggregated_path.exists():
        part2_enhanced_results = pd.read_csv(enhanced_aggregated_path)
        display(part2_enhanced_results)
    else:
        print('Enhanced Part 2 results were not found. Set RUN_ENHANCED_PART2 = True to train missing runs.')

for figure_key in ['all_points_figure', 'mean_by_seed_figure']:
    figure_path = Path(part2_enhanced_output_paths[figure_key])
    if figure_path.exists():
        display(Image(filename=str(figure_path)))
    else:
        print(f'Enhanced Part 2 figure not found yet: {figure_path}')


## Results

In [ ]:
results_path = Path(output_paths['aggregated_results'])

if results_path.exists():
    part2_results = pd.read_csv(results_path)
    if 'regular_part1' not in set(part2_results.get('ablation_name', [])) and not part1_baseline_aggregated.empty:
        part2_results = pd.concat([part1_baseline_aggregated, part2_results], ignore_index=True, sort=False)
    display(part2_results.sort_values(['tiles_per_side', 'ablation_name']))
else:
    print('Part 2 aggregated results were not found. Set RUN_TRAINING = True and run the training cell.')
    if part1_baseline_aggregated.empty:
        print('Part 1 ResNet-18 baseline results were also not found; run Part 1 first to include regular_part1.')

In [ ]:
import src.utils.results_overview as results_overview

intermediate_dir = Path(output_paths['intermediate_figures_dir'])
for figure_path in results_overview.latest_intermediate_figures(intermediate_dir, prefix='part2_'):
    display(Image(filename=str(figure_path)))

figure_path = Path(output_paths['figure'])
if figure_path.exists():
    display(Image(filename=str(figure_path)))
else:
    print('Part 2 ablation figure has not been generated yet.')


In [ ]:
# Export this saved notebook to PDF. Save the notebook before running this cell,
# because nbconvert reads the on-disk .ipynb file rather than unsaved editor state.
import importlib

import src.utils.notebook_setup as notebook_setup

notebook_setup = importlib.reload(notebook_setup)
notebook_path = ROOT / 'src' / 'notebooks' / 'part2_solution.ipynb'
export_dir = ROOT / 'outputs' / 'notebooks'
notebook_setup.export_notebook_to_pdf(notebook_path, export_dir)
